### Installation and Setup ###

In [ ]:
# Setup tools to avoid package conflicts
!pip install -U pip setuptools
!bash install_packages.sh

In [9]:
# Just checks devices available
import accelerate as acc
import wandb as wb
import torch
import transformers
import sentencepiece
import numpy as np
import google.protobuf  # Should import without error
print(np.__version__)  # Should print 1.26.4
print(torch.__version__)  # Should print 2.2.1
print(google.protobuf.__version__)  # Should print 4.25.3
print(torch.cuda.is_available())  # True
print(torch.cuda.device_count())  # 2
print(acc.__version__)
print(wb.__version__)

1.26.4
2.4.0+cu121
4.25.3
True
2
0.28.0
0.16.0


### Wandb Logging Setup ###

In [12]:
# Wandb logging setup
import wandb

# Log in to wandb (only required once per environment/session)
wandb.login()

# Initialize wandb for this training run
wandb.init(
    project="mistral-rpi-finetune",
    name="mistral-lora-run-v1",  # change the run name if needed
    reinit=True,
    config={
        "batch_size": 2,
        "learning_rate": 2e-4,
        "dataset": "train_mistral_jsonl.jsonl",
        "max_epochs": 3,
        "gradient_accumulation_steps": 4,
    },
)


wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········


wandb: Appending key for api.wandb.ai to your netrc file: /home/ucloud/.netrc
wandb: Currently logged in as: alidarlatif4 (alidarlatif4-sdu-mechatronics). Use `wandb login --relogin` to force relogin
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


### Training Script ###

In [ ]:
# Complete Training Script with SFT Trainer and all parameters aligned

# Step 1: Necassary Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset
import os

# Hugging face authentication
from huggingface_hub import login
# Replace with your actual token from huggingface.co/settings/tokens
login(token="YOUR_HF_TOKEN")

def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable} / {total} ({100 * trainable / total:.2f}%)")

# Step 2: Verify CUDA
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, GPUs: {torch.cuda.device_count()}")

# Step 3: Load and split dataset
dataset = load_dataset("json", data_files="train_mistral_jsonl.jsonl", split="train")
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Step 4: Preprocess dataset
def format_prompt(example):
    return {"text": f"<s>[INST] {example['prompt']} [/INST] {example['completion']}</s>"}

train_dataset = train_dataset.map(format_prompt)
eval_dataset = eval_dataset.map(format_prompt)

# Step 5: Set 4-bit quantization config (QLORA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4"
)

# Step 6: Load model
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Step 7: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Step 8: Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# Step 9: Set LoRA config and get PEFT model for training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()        # Reduces memory usage
model.enable_input_require_grads()           # Required for QLoRA to compute LoRA gradients

# Step 10: Set training arguments
training_args = TrainingArguments(
    output_dir=f"./results/mistral_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduced for longer sequences
    gradient_accumulation_steps=4,  # Adjusted to maintain batch size
    learning_rate=2e-4,
    weight_decay = 0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    evaluation_strategy = "steps",
    save_strategy = "steps",
    logging_steps = 10,
    eval_steps = 100,
    save_steps = 300,
    save_total_limit=2,
    bf16 = True, 
    logging_dir = os.path.join(f"./results/mistral_finetuned", "logs"),
    report_to="wandb", # or wandb
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    group_by_length=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Step 11: Set up trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=8192
)

# Step 12: Train the model
trainer.train()

# Step 13: Save the model and tokenizer
model.save_pretrained("./mistral_finetuned_final")
tokenizer.save_pretrained("./mistral_finetuned_final")

with open("./mistral_finetuned/training_config.txt", "w") as f:
    f.write(str(training_args))

with open("./mistral_finetuned/lora_config.txt", "w") as f:
    f.write(str(lora_config))


PyTorch: 2.4.0+cu121, CUDA: 12.1, GPUs: 2


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/opt/conda/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss


### Inference - Batch ###

In [ ]:
#!/usr/bin/env python3
"""
Generate and Compare Responses from Base and Fine-Tuned Mistral Models

This script:
1. Loads prompts from a JSON file
2. Generates code responses using both base and fine-tuned Mistral Coder models
3. Saves results to:
   - Generates Individual C files (one per prompt)
   - A consolidated JSON file with all results

Created: 2025-05-17 02:44:17
Author: Ali
"""

import json
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, AutoPeftModelForCausalLM, PeftConfig
import re
from datetime import datetime
import getpass
from tqdm import tqdm

import gc
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Configuration
INPUT_JSON = "Mistral_Test_Train_Samples_Base_Evaluation_Train2_20thMay2025.json"
OUTPUT_JSON = "Mistral_Test_Train_Responses_Evaluation_Train2_20thMay2025.json"
OUTPUT_DIR = "Mistral_Evaluation_Round2_20thMay2025"
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
FINETUNED_MODEL_PATH = "./results/mistral_finetuned/best_model"
RANDOM_SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
USERNAME = "Ali Hassan"
EXECUTION_DEVICE = "CPU"

def setup_directory(directory):
    os.makedirs(directory, exist_ok=True)

def load_json(file_path):
    print(f"Loading data from {file_path}...")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            examples = json.load(f)
            if not isinstance(examples, list):
                raise ValueError("Expected a list of JSON objects in the file.")
        except json.JSONDecodeError as e:
            raise ValueError(f"Error parsing JSON: {e}")
    print(f"Successfully loaded {len(examples)} examples.")
    return examples

def load_mistral_models():
    print("\n2. Loading fine-tuned model...")
    start_time = time.time()

    # Load adapter config
    peft_config = PeftConfig.from_pretrained(FINETUNED_MODEL_PATH)

    # Load the base model separately
    base_model = AutoModelForCausalLM.from_pretrained(
        peft_config.base_model_name_or_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Load the adapter on top of the base model
    finetuned_model = PeftModel.from_pretrained(
        base_model,
        FINETUNED_MODEL_PATH,
        torch_dtype=torch.bfloat16
    )

    # Tokenizer
    finetuned_tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

    print(f"   Fine-tuned model loaded in {time.time() - start_time:.2f} seconds")
    return finetuned_model, finetuned_tokenizer

def generate_mistral_response(model, tokenizer, prompt, prefix="<s>[INST]", suffix="[/INST]", cal_max_new_tokens=2048):
    formatted_prompt = f"{prefix} {prompt} {suffix}"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        start_time = time.time()
        outputs = model.generate(
            **inputs,
            max_new_tokens=cal_max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
        generation_time = time.time() - start_time

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    try:
        reply = full_response.split(suffix)[1].strip()
        if reply.endswith("</s>"):
            reply = reply[:-4].strip()
    except:
        reply = full_response

    clean_response = extract_clean_code(reply)
    print(f"   Generation time: {generation_time:.2f} seconds")
    return full_response, clean_response, generation_time

def extract_clean_code(response):
    code_match = re.search(r"```c\n(.*?)```", response, re.DOTALL)
    if not code_match:
        code_match = re.search(r"```(.*?)```", response, re.DOTALL)
    if code_match:
        return code_match.group(1).strip()

    lines = response.split('\n')
    code_lines = []
    in_code = False
    explanation_patterns = [
        r'^(Here\'s|This|Let\'s|Now|First|The code|I\'ll|To|We|That\'s|This will)',
        r'.*\b(explains|provides|shows|demonstrates|illustrates)\b.*',
        r'^In this code',
        r'^This program',
    ]
    for line in lines:
        stripped = line.strip()
        if not stripped and not in_code:
            continue
        if any(re.match(pattern, stripped, re.IGNORECASE) for pattern in explanation_patterns) and not in_code:
            continue
        if re.match(r'^(#include|int\s+main|void\s+main|void\s+\w+|int\s+\w+|float\s+\w+|char\s+\w+|printf|scanf|for\s*\(|while\s*\(|if\s*\(|return\b)', stripped):
            in_code = True
            code_lines.append(line)
        elif in_code or (stripped and not stripped.endswith('.') and not stripped.endswith('?')):
            code_lines.append(line)

    return '\n'.join(code_lines) if code_lines else response

def save_to_file(example_id, prompt, source, finetuned_response, output_dir):
    finetuned_file = os.path.join(output_dir, f"example_fineTunedMistral_id_{example_id}_{source}.c")

    with open(finetuned_file, "w", encoding="utf-8") as f:
        f.write(f"""/*
 * Fine-Tuned Mistral Response for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

{finetuned_response}
""")

    return finetuned_file

def main():
    print(f"=== Mistral Coder Response Generation ===")
    print(f"Date/Time: {TIMESTAMP}")
    print(f"User: {USERNAME}")
    torch.manual_seed(RANDOM_SEED)
    setup_directory(OUTPUT_DIR)

    examples = load_json(INPUT_JSON)
    total_examples = len(examples)

    completed_results = []
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                completed_results = json.load(f)
                print(f"🔁 Resuming from checkpoint. Already processed: {len(completed_results)} examples.")
        except Exception as e:
            print(f"⚠️ Error reading checkpoint: {e}")
            completed_results = []

    start_index = len(completed_results)
    finetuned_model, finetuned_tokenizer = load_mistral_models()

    start_time = time.time()

    for i, example in enumerate(tqdm(examples[start_index:], desc="🧠 Generating", unit="example", ncols=100), start=start_index):
        example_id = example["id"]
        prompt = example["input"]
        source = example["data_source"]
        code_reference = example["output"]

        output_len = len(code_reference)
        estimated_tokens = int((output_len / 4) * 2) if output_len > 0 else 2048
        cal_max_new_tokens = min(8192, max(128, estimated_tokens))

        tqdm.write(f"\n[{i+1}/{total_examples}] Processing prompt {example_id} (Source: {source}) (Allowed Tokens: {cal_max_new_tokens})")

        tqdm.write(f"   Generating fine-tuned Mistral response...")
        finetuned_response, finetuned_code, finetuned_time = generate_mistral_response(finetuned_model, finetuned_tokenizer, prompt, cal_max_new_tokens=cal_max_new_tokens)

        save_to_file(example_id, prompt, source, finetuned_code, OUTPUT_DIR)
        tqdm.write(f"   Saved to file set")

        example["finetuned_response"] = finetuned_response
        example["finetuned_code"] = finetuned_code
        example["finetuned_execution_time"] = finetuned_time
        example["allowed_max_new_tokens"] = cal_max_new_tokens
        example["code_gen_timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        completed_results.append(example)

        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(completed_results, f, indent=2)

        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (total_examples - i - 1)
        eta_min, eta_sec = divmod(int(remaining), 60)
        tqdm.write(f"⏱️ Time/iter: {avg_time:.2f}s | ETA: {eta_min}m {eta_sec}s")

    print("\n=== Mistral Inference/Code Generation Complete ===")
    print(f"- Processed {total_examples} prompts")
    print(f"- Individual C files saved to: {OUTPUT_DIR}/")
    print(f"- Consolidated results saved to: {OUTPUT_JSON}")
    print(f"- Generation timestamp: {TIMESTAMP}")
    print(f"- Generated by: {USERNAME}")

if __name__ == "__main__":
    main()

=== Mistral Coder Response Generation ===
Date/Time: 2025-05-20 18:03:24
User: Ali Hassan
Loading data from Mistral_Test_Train_Samples_Base_Evaluation_Train2_20thMay2025.json...
Successfully loaded 40 examples.
🔁 Resuming from checkpoint. Already processed: 1 examples.

2. Loading fine-tuned model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

   Fine-tuned model loaded in 8.74 seconds


🧠 Generating:   0%|                                                    | 0/39 [00:00<?, ?example/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



[2/40] Processing prompt 2205 (Source: test) (Allowed Tokens: 4486)
   Generating fine-tuned Mistral response...


...

   Generation time: 97.87 seconds
   Saved to file set
⏱️ Time/iter: 130.65s | ETA: 30m 29s

[27/40] Processing prompt 749 (Source: train) (Allowed Tokens: 3896)
   Generating fine-tuned Mistral response...


🧠 Generating:  67%|████████████████████████████              | 26/39 [59:25<31:52, 147.11s/example]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Generation time: 169.08 seconds
   Saved to file set
⏱️ Time/iter: 132.07s | ETA: 28m 36s

[28/40] Processing prompt 1380 (Source: train) (Allowed Tokens: 1667)
   Generating fine-tuned Mistral response...


🧠 Generating:  69%|███████████████████████████▋            | 27/39 [1:00:38<24:57, 124.82s/example]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Generation time: 72.78 seconds
   Saved to file set
⏱️ Time/iter: 129.95s | ETA: 25m 59s

[29/40] Processing prompt 5184 (Source: train) (Allowed Tokens: 5165)
   Generating fine-tuned Mistral response...


### Inference - Variations ###

In [ ]:
#!/usr/bin/env python3
"""
Generate Response of fine tuned mistral to variations (Train iteration 2)

This script:
1. Loads prompts from a JSON file
2. Generates code responses using both base and fine-tuned Mistral Coder models
3. Saves results to:
   - Generates Individual C files (one per prompt)
   - A consolidated JSON file with all results

Created: 2025-05-20 20:00:17
Author: Ali
"""

import json
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM, PeftConfig
import re
from datetime import datetime
import getpass
from tqdm import tqdm

# Configuration
INPUT_JSON = "Mistral_Train2_Prompts_30_Variations_Evaluation_20thMay2025.json"
OUTPUT_JSON = "Mistral_Train2_Responses_Variations_Evaluation_20thMay2025.json"
OUTPUT_DIR = "Mistral_Train2_Evaluation_Round2_Variations_18thMay2025"
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
FINETUNED_MODEL_PATH = "./results/mistral_finetuned/best_model"
RANDOM_SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
USERNAME = "Ali Hassan"
EXECUTION_DEVICE = "CPU"

def setup_directory(directory):
    os.makedirs(directory, exist_ok=True)

def load_json(file_path):
    """Load examples from a JSON file containing a list of objects with subid != '0'."""
    print(f"Loading data from {file_path}...")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            examples = json.load(f)
            if not isinstance(examples, list):
                raise ValueError("Expected a list of JSON objects in the file.")
        except json.JSONDecodeError as e:
            raise ValueError(f"Error parsing JSON: {e}")

    print(f"Loaded {len(examples)} entries from the file.")
    return examples

def load_mistral_models():
    print("\n2. Loading fine-tuned model...")
    start_time = time.time()

    # Load adapter config
    peft_config = PeftConfig.from_pretrained(FINETUNED_MODEL_PATH)

    # Load the base model separately
    base_model = AutoModelForCausalLM.from_pretrained(
        peft_config.base_model_name_or_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Load the adapter on top of the base model
    finetuned_model = PeftModel.from_pretrained(
        base_model,
        FINETUNED_MODEL_PATH,
        torch_dtype=torch.bfloat16
    )

    # Tokenizer
    finetuned_tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

    print(f"   Fine-tuned model loaded in {time.time() - start_time:.2f} seconds")
    return finetuned_model, finetuned_tokenizer

def generate_mistral_response(model, tokenizer, prompt, prefix="<s>[INST]", suffix="[/INST]", cal_max_new_tokens=2048):
    formatted_prompt = f"{prefix} {prompt} {suffix}"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        start_time = time.time()
        outputs = model.generate(
            **inputs,
            max_new_tokens=cal_max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
        generation_time = time.time() - start_time

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    try:
        reply = full_response.split(suffix)[1].strip()
        if reply.endswith("</s>"):
            reply = reply[:-4].strip()
    except:
        reply = full_response

    clean_response = extract_clean_code(reply)
    print(f"   Generation time: {generation_time:.2f} seconds")
    return full_response, clean_response, generation_time

def extract_clean_code(response):
    code_match = re.search(r"```c\n(.*?)```", response, re.DOTALL)
    if not code_match:
        code_match = re.search(r"```(.*?)```", response, re.DOTALL)
    if code_match:
        return code_match.group(1).strip()

    lines = response.split('\n')
    code_lines = []
    in_code = False
    explanation_patterns = [
        r'^(Here\'s|This|Let\'s|Now|First|The code|I\'ll|To|We|That\'s|This will)',
        r'.*\b(explains|provides|shows|demonstrates|illustrates)\b.*',
        r'^In this code',
        r'^This program',
    ]
    for line in lines:
        stripped = line.strip()
        if not stripped and not in_code:
            continue
        if any(re.match(pattern, stripped, re.IGNORECASE) for pattern in explanation_patterns) and not in_code:
            continue
        if re.match(r'^(#include|int\s+main|void\s+main|void\s+\w+|int\s+\w+|float\s+\w+|char\s+\w+|printf|scanf|for\s*\(|while\s*\(|if\s*\(|return\b)', stripped):
            in_code = True
            code_lines.append(line)
        elif in_code or (stripped and not stripped.endswith('.') and not stripped.endswith('?')):
            code_lines.append(line)

    return '\n'.join(code_lines) if code_lines else response

def save_to_file(example_id, prompt, source, finetuned_response, output_dir):
    finetuned_file = os.path.join(output_dir, f"example_fineTunedMistral_id_{example_id}_{source}.c")

    with open(finetuned_file, "w", encoding="utf-8") as f:
        f.write(f"""/*
            * Fine-Tuned Mistral Response for prompt ID: {example_id}
            * Source: {source}
            * Generated on: {TIMESTAMP}
            * Generated by: {USERNAME}
            */

            /* Original prompt:
            {prompt}
            */

            {finetuned_response}
            """)

    return finetuned_file

def main():
    print(f"=== Mistral Coder Response Generation ===")
    print(f"Date/Time: {TIMESTAMP}")
    print(f"User: {USERNAME}")
    torch.manual_seed(RANDOM_SEED)
    setup_directory(OUTPUT_DIR)

    examples = load_json(INPUT_JSON)
    total_examples = len(examples)

    completed_results = []
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                completed_results = json.load(f)
                print(f"🔁 Resuming from checkpoint. Already processed: {len(completed_results)} examples.")
        except Exception as e:
            print(f"⚠️ Error reading checkpoint: {e}")
            completed_results = []

    start_index = len(completed_results)
    finetuned_model, finetuned_tokenizer = load_mistral_models()

    start_time = time.time()

    for i, example in enumerate(tqdm(examples[start_index:], desc="🧠 Generating", unit="example", ncols=100), start=start_index):
        example_id = example["id"]
        example_subid = example["subid"]
        prompt = example["input"]
        source = example["data_source"]
        code_reference = example["output"]

        if example_subid != "0":
            output_len = len(code_reference)
            estimated_tokens = int((output_len / 4) * 2) if output_len > 0 else 2048
            cal_max_new_tokens = min(8192, max(128, estimated_tokens))

            tqdm.write(f"\n[{i+1}/{total_examples}] Processing prompt {example_id} (Source: {source}) (Allowed Tokens: {cal_max_new_tokens})")
            tqdm.write(f"   Generating fine-tuned Mistral response...")
            finetuned_response, finetuned_code, finetuned_time = generate_mistral_response(finetuned_model, finetuned_tokenizer, prompt, cal_max_new_tokens=cal_max_new_tokens)

            save_to_file(example_id, prompt, source, finetuned_code, OUTPUT_DIR)
            tqdm.write(f"   Saved to file set")

            example["finetuned_response"] = finetuned_response
            example["finetuned_code"] = finetuned_code
            example["finetuned_execution_time"] = finetuned_time
            example["allowed_max_new_tokens"] = cal_max_new_tokens
            example["code_gen_timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        completed_results.append(example)

        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(completed_results, f, indent=2)

        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (total_examples - i - 1)
        eta_min, eta_sec = divmod(int(remaining), 60)
        tqdm.write(f"⏱️ Time/iter: {avg_time:.2f}s | ETA: {eta_min}m {eta_sec}s")

    print("\n=== Mistral Inference/Code Generation Complete ===")
    print(f"- Processed {total_examples} prompts")
    print(f"- Individual C files saved to: {OUTPUT_DIR}/")
    print(f"- Consolidated results saved to: {OUTPUT_JSON}")
    print(f"- Generation timestamp: {TIMESTAMP}")
    print(f"- Generated by: {USERNAME}")

if __name__ == "__main__":
    main()


### Inference - Pass@K ###

In [ ]:
#!/usr/bin/env python3
"""
Generate Responses from Base and Fine-Tuned Mistral Models for pass@k evaluation

This script:
1. Loads prompts from a JSON file
2. Generates code responses using both base and fine-tuned Mistral Coder models
3. Saves results to:
   - Generates Individual C files (one per prompt)
   - A consolidated JSON file with all results

Created: 2025-05-20 21:00:17
Author: Ali Hassan
"""

import json
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM
import re
from datetime import datetime
import getpass
from tqdm import tqdm

# Configuration
INPUT_JSON = "Mistral_Train2_Prompts_50_Pass@5_Evaluation_20thMay2025.json"
OUTPUT_JSON = "Mistral_Train2_Response_Pass@5_Evaluation_20thMay2025.json"
OUTPUT_DIR = "Mistral_Evaluation_Train2_pass_k_20thMay2025"
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
FINETUNED_MODEL_PATH = "./results/mistral_finetuned/best_model"
RANDOM_SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
USERNAME = "Ali Hassan"
EXECUTION_DEVICE = "CPU"

def setup_directory(directory):
    os.makedirs(directory, exist_ok=True)

def load_json(file_path):
    print(f"Loading data from {file_path}...")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            examples = json.load(f)
            if not isinstance(examples, list):
                raise ValueError("Expected a list of JSON objects in the file.")
        except json.JSONDecodeError as e:
            raise ValueError(f"Error parsing JSON: {e}")
    print(f"Successfully loaded {len(examples)} examples.")
    return examples

def load_mistral_models():
    print("\n2. Loading fine-tuned model...")
    start_time = time.time()

    # Load adapter config
    peft_config = PeftConfig.from_pretrained(FINETUNED_MODEL_PATH)

    # Load the base model separately
    base_model = AutoModelForCausalLM.from_pretrained(
        peft_config.base_model_name_or_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Load the adapter on top of the base model
    finetuned_model = PeftModel.from_pretrained(
        base_model,
        FINETUNED_MODEL_PATH,
        torch_dtype=torch.bfloat16
    )

    # Tokenizer
    finetuned_tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

    print(f"   Fine-tuned model loaded in {time.time() - start_time:.2f} seconds")
    return finetuned_model, finetuned_tokenizer

def generate_mistral_response(model, tokenizer, prompt, prefix="<s>[INST]", suffix="[/INST]", cal_max_new_tokens=2048):
    formatted_prompt = f"{prefix} {prompt} {suffix}"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        start_time = time.time()
        outputs = model.generate(
            **inputs,
            max_new_tokens=cal_max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
        generation_time = time.time() - start_time

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    try:
        reply = full_response.split(suffix)[1].strip()
        if reply.endswith("</s>"):
            reply = reply[:-4].strip()
    except:
        reply = full_response

    clean_response = extract_clean_code(reply)
    print(f"   Generation time: {generation_time:.2f} seconds")
    return full_response, clean_response, generation_time

def extract_clean_code(response):
    code_match = re.search(r"```c\n(.*?)```", response, re.DOTALL)
    if not code_match:
        code_match = re.search(r"```(.*?)```", response, re.DOTALL)
    if code_match:
        return code_match.group(1).strip()

    lines = response.split('\n')
    code_lines = []
    in_code = False
    explanation_patterns = [
        r'^(Here\'s|This|Let\'s|Now|First|The code|I\'ll|To|We|That\'s|This will)',
        r'.*\b(explains|provides|shows|demonstrates|illustrates)\b.*',
        r'^In this code',
        r'^This program',
    ]
    for line in lines:
        stripped = line.strip()
        if not stripped and not in_code:
            continue
        if any(re.match(pattern, stripped, re.IGNORECASE) for pattern in explanation_patterns) and not in_code:
            continue
        if re.match(r'^(#include|int\s+main|void\s+main|void\s+\w+|int\s+\w+|float\s+\w+|char\s+\w+|printf|scanf|for\s*\(|while\s*\(|if\s*\(|return\b)', stripped):
            in_code = True
            code_lines.append(line)
        elif in_code or (stripped and not stripped.endswith('.') and not stripped.endswith('?')):
            code_lines.append(line)

    return '\n'.join(code_lines) if code_lines else response

def save_to_file(example_id, prompt, source, finetuned_response, output_dir):
    finetuned_file = os.path.join(output_dir, f"example_fineTunedMistral_id_{example_id}_{source}.c")

    with open(finetuned_file, "w", encoding="utf-8") as f:
        f.write(f"""/*
            * Fine-Tuned Mistral Response for prompt ID: {example_id}
            * Source: {source}
            * Generated on: {TIMESTAMP}
            * Generated by: {USERNAME}
            */

            /* Original prompt:
            {prompt}
            */

            {finetuned_response}
            """)

    return finetuned_file

def main():
    print(f"=== Mistral Coder Response Generation ===")
    print(f"Date/Time: {TIMESTAMP}")
    print(f"User: {USERNAME}")
    torch.manual_seed(RANDOM_SEED)
    setup_directory(OUTPUT_DIR)

    examples = load_json(INPUT_JSON)
    total_examples = len(examples)

    completed_results = []
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                completed_results = json.load(f)
                print(f"🔁 Resuming from checkpoint. Already processed: {len(completed_results)} examples.")
        except Exception as e:
            print(f"⚠️ Error reading checkpoint: {e}")
            completed_results = []

    start_index = len(completed_results)
    finetuned_model, finetuned_tokenizer = load_mistral_models()

    start_time = time.time()

    for i, example in enumerate(tqdm(examples[start_index:], desc="🧠 Generating", unit="example", ncols=100), start=start_index):
        example_id = example["id"]
        example_subid = example["subid"]
        prompt = example["input"]
        source = example["data_source"]
        code_reference = example["output"]

        if example_subid != "0":
            output_len = len(code_reference)
            estimated_tokens = int((output_len / 4) * 2) if output_len > 0 else 2048
            cal_max_new_tokens = min(8192, max(128, estimated_tokens))

            tqdm.write(f"\n[{i+1}/{total_examples}] Processing prompt {example_id} (Source: {source}) (Allowed Tokens: {cal_max_new_tokens})")

            tqdm.write(f"   Generating fine-tuned Mistral response...")
            finetuned_response, finetuned_code, finetuned_time = generate_mistral_response(finetuned_model, finetuned_tokenizer, prompt, cal_max_new_tokens=cal_max_new_tokens)

            save_to_file(example_id, prompt, source, finetuned_code, OUTPUT_DIR)
            tqdm.write(f"   Saved to file set")

            example["finetuned_response"] = finetuned_response
            example["finetuned_code"] = finetuned_code
            example["finetuned_execution_time"] = finetuned_time
            example["allowed_max_new_tokens"] = cal_max_new_tokens
            example["evaluation_datetime"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        completed_results.append(example)

        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(completed_results, f, indent=2)

        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (total_examples - i - 1)
        eta_min, eta_sec = divmod(int(remaining), 60)
        tqdm.write(f"⏱️ Time/iter: {avg_time:.2f}s | ETA: {eta_min}m {eta_sec}s")

    print("\n=== Mistral Inference/Code Generation Complete ===")
    print(f"- Processed {total_examples} prompts")
    print(f"- Individual C files saved to: {OUTPUT_DIR}/")
    print(f"- Consolidated results saved to: {OUTPUT_JSON}")
    print(f"- Generation timestamp: {TIMESTAMP}")
    print(f"- Generated by: {USERNAME}")

if __name__ == "__main__":
    main()
